In [1]:
import argparse

import torch

from model.encoder_standalone_cpu import Encoder, molecule_to_latent
from utils.config import load_config
from utils.data_loading import process_sdf_files_to_list

/home/teaching/miniconda3/envs/MolFLAE2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument('--sdf_folder', type=str,default='data/latent_experiment/val')
parser.add_argument('--output_folder', type=str,default='latent_experiment/ex1/output')
parser.add_argument('--ckpt_path', type=str,default='ckpt-zinc9M/model-epoch=24-val_loss=3.40.ckpt')
parser.add_argument('--config', type=str, default='config.yaml')
parser.add_argument('--batch_size', type=int, default=100)
parser.add_argument('--device', type=str, default='cpu')

args, unknown = parser.parse_known_args()

In [3]:
from model.train_loop import TrainLoopCharges

# Extract encoder configuration from the config file
config = load_config(args.config)
net_config = config["decoder_config_charge"]["net_config"]

train_loop = TrainLoopCharges(config)
train_loop.configure_optimizers()


Encoder params: 2.96M
KL layer params: 0.01M
Decoder params: 2.84M
Of which 16641 are charge head params.
total params: 5.81M



{'optimizer': Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.95, 0.999)
     capturable: False
     decoupled_weight_decay: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0005
     maximize: False
     weight_decay: 0
 ),
 'lr_scheduler': {'scheduler': <torch.optim.lr_scheduler.ReduceLROnPlateau at 0x7969be3b4370>,
  'monitor': 'val_loss',
  'interval': 'epoch',
  'frequency': 0.25}}

In [4]:
# Load the molecules
mols = process_sdf_files_to_list(args.sdf_folder)

# Build a numbered atom batch
all_h = torch.cat([entry['h'] for entry in mols], dim=0)
all_x = torch.cat([entry['x'] for entry in mols], dim=0)
all_batch = []
current_index = 0
for entry in mols:
    atom_num = entry['atom_num']
    all_batch += [current_index] * atom_num
    current_index += 1
all_batch = torch.tensor(all_batch, dtype=torch.long)

Processing molecules: 100%|██████████| 932/932 [00:00<00:00, 1437.18mol/s]


In [ ]:
import wandb

wandb.init()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/teaching/.netrc.
wandb: Currently logged in as: somique (somique-university-of-oxford) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [6]:
from torch_geometric.data import Data, Batch
import torch

# Aggregate molecular info into Data objects 
mol1 = Data(
    h=torch.tensor([[6], [7], [8]], dtype=torch.long),
    x=torch.randn(3, 3),
    charges=torch.tensor([[-0.2], [-0.5], [0.7]], dtype=torch.float32),
)

mol2 = Data(
    h=torch.tensor([[6], [6], [7], [15]], dtype=torch.long),
    x=torch.randn(4, 3),
    charges=torch.tensor([[0.1], [-0.1], [-0.3], [0.3]], dtype=torch.float32),
)

batch = Batch.from_data_list([mol1, mol2])
train_loop.configure_optimizers()

train_loop.optim.zero_grad()
loss = train_loop.training_step(batch, None)
loss.backward()
print("charge_head grads exist?:", any(p.grad is not None for p in train_loop.decoder.charge_head.parameters()))

/home/teaching/miniconda3/envs/MolFLAE2/lib/python3.10/site-packages/pytorch_lightning/core/module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


charge_head grads exist?: True


## Now, read in a molecule and feed it to the model

In [7]:
from  rdkit import Chem

sdf_path = "../csd_mol.sdf"

mols = []
with Chem.SDMolSupplier(sdf_path, removeHs=False) as suppl:
    for mol in suppl:
        if mol is not None: mols.append(mol)

In [8]:
NAME_TO_ATOM_NUM = {
    'H' : 1,
    'C' : 6,
    'N' : 7,
    'O' : 8,
    'F' : 9,
    'P' : 15,
    'Cl': 17,
    'Ru': 44,
    'W' : 74,
}

MAP_ATOM_TYPE_ONLY_TO_INDEX = {
    6: 0,
    7: 1,
    8: 2,
    9: 3,
    15: 4,
    16: 5,
    17: 6,
    35: 7,
    53: 8,
}

In [9]:
mol = mols[0]
conf = mol.GetConformer()

coords = []
atom_nums = []

for atom in mol.GetAtoms():
    atom_nums.append(NAME_TO_ATOM_NUM[atom.GetSymbol()])

    idx = atom.GetIdx()
    pos = conf.GetAtomPosition(idx)
    coords.append([pos.x, pos.y, pos.z])

charges = [-0.3304679521787277, -0.0358678079188456, -0.1442257517466099,
            0.2285565734887312,  0.0586682434373685, -0.0264170273286043,
            -0.0837763695657814, -0.1230457889803089,  0.1304131565477271,
            -0.1940540966668532,  0.2329050026107839, -0.119552504617812,
            -0.0746443946518365, -0.1000633731762596, -0.0049090544087934,
            -0.1147671245885875, -0.1685660858751479, -0.2306175622603055,
            -1.2551235641718237,  1.964700399196359,   0.0074987328705483,
            -0.4339798386658052,  0.2751566748722292,  0.0993989211143413,
            0.0615391021021094, -0.2150075503226345,  0.1336200130230081,
            0.0071810434027228,  0.0564882354510841,  0.349043663442311,
            0.0137196565640887,  0.0137196565640887,  0.2676227991912004,
            0.2676227991912004,  0.4481288505884815, -0.4561384072638367,
            -0.4561384072638367, -0.4561384072638367,  0.1550644037713899,
            0.2524531414864733]

In [10]:
import numpy as np

atom_nums = np.array(atom_nums)  # or np.array(h)
coords = np.array(coords)
charges = np.array(charges)

mask = atom_nums != 1

coords_no_H = coords[mask]
atom_nums_no_H = atom_nums[mask]
charges_no_H = charges[mask]

In [11]:
from torch_geometric.data import Data, Batch
import torch

# Pack molecular information into Data objects
mol1 = Data(
    h=torch.tensor(atom_nums_no_H, dtype=torch.long).unsqueeze(1),
    x=torch.tensor(coords_no_H, dtype=torch.float32),
    charges=torch.tensor(charges_no_H, dtype=torch.float32).unsqueeze(1),
)
mol2 = Data(
    h=torch.tensor(atom_nums_no_H, dtype=torch.long).unsqueeze(1),
    x=torch.tensor(coords_no_H, dtype=torch.float32),
    charges=torch.tensor(charges_no_H, dtype=torch.float32).unsqueeze(1),
)

# Buld the minibatch
batch = Batch.from_data_list([mol1, mol2])

# Do a training step
train_loop.optim.zero_grad()
train_loss = train_loop.training_step(batch, None)
train_loss.backward()
print("charge_head grads exist?:", any(p.grad is not None for p in train_loop.decoder.charge_head.parameters()))

# Do a validation step
with torch.no_grad():
    valid_batch = batch  # Will need to change this
    valid_batch_idx = 0
    valid_loss = train_loop.validation_step(valid_batch, valid_batch_idx )


charge_head grads exist?: True


Val: 100%|██████████| 100/100 [00:18<00:00,  5.38it/s]


In [12]:
## To do: have a big loop that takes in a batch e.g. as a pytorch Dataset or similar

In [19]:
import numpy as np

def absorb_hydrogen_charges_to_heavy_atoms(mol, charges):
    """
    Move each H charge onto its single bonded heavy atom.

    mol: RDKit Mol with explicit hydrogens
    charges: array-like, length mol.GetNumAtoms()
    returns: np.ndarray, length mol.GetNumAtoms(), with H charges zeroed
    """
    charges = np.asarray(charges, dtype=np.float32).copy()

    assert len(charges) == mol.GetNumAtoms()

    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() != 1:
            continue

        h_idx = atom.GetIdx()
        neighbors = atom.GetNeighbors()

        if len(neighbors) != 1:
            raise ValueError(f"Hydrogen atom {h_idx} has {len(neighbors)} neighbors")

        heavy_idx = neighbors[0].GetIdx()
        charges[heavy_idx] += charges[h_idx]
        charges[h_idx] = 0.0

    return charges

def mol_to_heavy_atom_data_with_charges(mol, charges, absorb_h=True):
    charges = np.asarray(charges, dtype=np.float32)
    assert len(charges) == mol.GetNumAtoms()

    if absorb_h:
        charges = absorb_hydrogen_charges_to_heavy_atoms(mol, charges)

    conf = mol.GetConformer()

    atom_nums = []
    coords = []
    heavy_charges = []

    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 1:
            continue  # skip H atoms

        idx = atom.GetIdx()
        pos = conf.GetAtomPosition(idx)

        atom_nums.append(atom.GetAtomicNum())
        coords.append([pos.x, pos.y, pos.z])
        heavy_charges.append(charges[idx])

    atom_nums = np.asarray(atom_nums, dtype=np.int64)
    coords = np.asarray(coords, dtype=np.float32)
    heavy_charges = np.asarray(heavy_charges, dtype=np.float32)

    return atom_nums, coords, heavy_charges  

In [30]:
import torch
import numpy as np
from torch_geometric.data import Data, Dataset
from rdkit import Chem

class RDKitChargeDataset(Dataset):
    def __init__(
            self,
            mols,
            charge_arrays,
            remove_hs=True,
            absorb_h_charges = True,
            allowed_atom_nums=None):
        
        super().__init__()

        if not len(mols) == len(charge_arrays):
            raise ValueError(f"Length of molecule array ({len(mols)}) doesn't match the length of the charges array  ({len(charges)}).")
        
        self.mols = mols
        if absorb_h_charges:
            for i in range(len(charge_arrays)):
                charge_arrays[i] = absorb_hydrogen_charges_to_heavy_atoms(mols[i], charge_arrays[i])
        self.charge_arrays = charge_arrays
        self.remove_hs = remove_hs
        self.allowed_atom_nums = (
            set(allowed_atom_nums)
            if allowed_atom_nums is not None
            else set(MAP_ATOM_TYPE_ONLY_TO_INDEX)
            )
        self.valid_indices = self._filter_valid_indices()

    def _filter_valid_indices(self):
        valid = []

        for i, (mol, q) in enumerate(zip(self.mols, self.charge_arrays)):
            if mol is None:
                continue

            if mol.GetNumConformers() == 0:
                continue

            if len(q) != mol.GetNumAtoms():
                continue

            atom_nums = np.array([atom.GetAtomicNum() for atom in mol.GetAtoms()])

            if self.remove_hs:
                atom_nums_used = atom_nums[atom_nums != 1]
            else:
                atom_nums_used = atom_nums

            if len(atom_nums_used) == 0:
                continue

            unsupported = set(atom_nums_used.tolist()) - self.allowed_atom_nums
            if unsupported:
                continue

            valid.append(i)

        return valid
    
    def len(self):
        return len(self.valid_indices)
    
    def get(self, idx):
        real_idx = self.valid_indices[idx]

        mol = self.mols[real_idx]
        q = np.asarray(self.charges[real_idx], dtype=np.float64)
        conf = mol.GetConformer()

        atom_nums = []
        coords = []
        for atom in mol.GetAtoms():
            # Parse the atoms object for atom number and coords
            atom_num = atom.GetAtomicNum()
            atom_idx = atom.GetIdx()
            pos = conf.GetAtomPosition(atom_idx)

            atom_nums.append(atom_num)
            coords.append([pos.x, pos.y, pos.z])

        atom_nums = np.asarray(atom_nums, dtype=np.int64)
        coords = np.asarray(coords, dtype=np.float32)

        if self.remove_hs:
            mask = atom_nums != 1
            atom_nums = atom_nums[mask]
            coords = coords[mask]
            q = q[mask]

        data = Data(
            h=torch.tensor(atom_nums, dtype=torch.long).view(-1, 1),
            x=torch.tensor(coords, dtype=torch.float32),
            charge_arrays=torch.tensor(q, dtype=torch.float32).view(-1, 1),
        )

        return data

In [31]:
sdf_path = "../csd_mol.sdf"

mols = []
with Chem.SDMolSupplier(sdf_path, removeHs=False) as suppl:
    for mol in suppl:
        if mol is not None: mols.append(mol)

charges = [-0.3304679521787277, -0.0358678079188456, -0.1442257517466099,
            0.2285565734887312,  0.0586682434373685, -0.0264170273286043,
            -0.0837763695657814, -0.1230457889803089,  0.1304131565477271,
            -0.1940540966668532,  0.2329050026107839, -0.119552504617812,
            -0.0746443946518365, -0.1000633731762596, -0.0049090544087934,
            -0.1147671245885875, -0.1685660858751479, -0.2306175622603055,
            -1.2551235641718237,  1.964700399196359,   0.0074987328705483,
            -0.4339798386658052,  0.2751566748722292,  0.0993989211143413,
            0.0615391021021094, -0.2150075503226345,  0.1336200130230081,
            0.0071810434027228,  0.0564882354510841,  0.349043663442311,
            0.0137196565640887,  0.0137196565640887,  0.2676227991912004,
            0.2676227991912004,  0.4481288505884815, -0.4561384072638367,
            -0.4561384072638367, -0.4561384072638367,  0.1550644037713899,
            0.2524531414864733]
charge_arrays = [np.asarray(charges, dtype=np.float64)]

In [32]:
from torch_geometric.loader import DataLoader

dataset = RDKitChargeDataset(
    mols=mols,
    charge_arrays=charge_arrays,
    remove_hs=True
)

loader = DataLoader(
    dataset,
    batch_size = 32,
    shuffle=True,
    num_workers=0
)

In [33]:
atom_nums, coords, heavy_q = mol_to_heavy_atom_data_with_charges(
    mol,
    charges,
    absorb_h=True,
)

print("original charge sum:", float(np.sum(charges)))
print("heavy implicit-H charge sum:", float(np.sum(heavy_q)))

original charge sum: 1.1102230246251565e-16
heavy implicit-H charge sum: -4.470348358154297e-08


In [2]:
import numpy as np
charges = np.asarray([3.5405681025332716, -1.5752280669325076, -0.297521073181375, 0.03877004723915478, -6.283471576280862, -1.7254395339954092, 2.7379176607677302, -0.9825111869838771, -0.8277278611335265, 0.5217005858791739, 2.2680227620573126, 0.05266665997966612, -0.042975020827093965, -0.6524882975536737, 5.00430195683664, 0.2344491686834249, -0.09021191418486159, -1.9196185316948668, -0.0004012937361070369, -0.0004012937361070369, -0.0004012937361070369],
                     dtype=np.float64)